# Account Security Posture Audit Notebook

Portable security posture audit for a new Snowflake environment, with an emphasis on data
egress and sensitive-data protection given a regulated/confidential data environment. Run top
to bottom. It surfaces:

1. **Network policies** — account-level and per-role/user enforcement, and the actual IP allow/block rules
2. **Storage & other integrations** — every integration that connects the account to something external (storage, API, notification), each a potential egress or trust boundary
3. **External stages** — where data can land outside Snowflake-managed storage, and who can write to them
4. **Data unloading / egress controls** — account parameters that govern whether `COPY INTO <location>` is even allowed, plus recent actual unload activity
5. **Data sharing inventory** — outbound shares (data leaving the account to other accounts) and inbound shares
6. **Masking & row access policy coverage** — whether policies exist at all, where they're applied, and columns that look sensitive by name but have no policy attached
7. **Authentication & session security** — MFA usage in practice, password/session policies, account-wide network policy enforcement
8. **A rolled-up recommendation summary**

### Prerequisites
- `SECURITYADMIN` or `ACCOUNTADMIN` for the policy/integration `SHOW` commands in this
  notebook — a plain `IMPORTED PRIVILEGES` role will see most `ACCOUNT_USAGE` data but may
  get empty results on some `SHOW` commands depending on grants.
- Nothing here is destructive — every cell reads. Nothing writes to the account.

### Scope note
This audits *configuration* — what's allowed and what's protected. It does not attempt to
detect an active exfiltration in progress; Section 4's unload-activity query is a starting
point for that kind of investigation, not a complete one.


In [ ]:
-- ============================================================
-- PARAMETERS
-- ============================================================
SET lookback_days = 30;
SET sensitive_column_patterns = '%SSN%,%SOCIAL_SEC%,%PATIENT%,%MRN%,%DOB%,%BIRTH%,%DIAGNOS%,%EMAIL%,%PHONE%,%ADDRESS%,%INSURANCE%,%MEDICAL%,%HEALTH%,%NAME%,%GENDER%,%RACE%,%ZIP%,%CREDIT_CARD%,%SSN_LAST4%';

SELECT $lookback_days AS lookback_days, $sensitive_column_patterns AS sensitive_column_patterns;


## 1. Network policies

An account with no network policy enforced at the account level means, by default, any
authenticated user can connect from anywhere on the internet. For a regulated data
environment, having *some* policies defined isn't the same as having them *enforced*.


In [ ]:
-- Is a network policy enforced at the account level?
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN ACCOUNT;


In [ ]:
-- All defined network policies
SHOW NETWORK POLICIES;


In [ ]:
-- Which roles/users have a network policy assigned directly (overriding the account default)
SELECT
    grantee_name,
    granted_on,
    privilege,
    name AS policy_name,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND granted_on = 'NETWORK POLICY'
ORDER BY grantee_name;


In [ ]:
# Pull the actual allow/block IP rules for every defined network policy. Looping SHOW/DESCRIBE
# per policy is fine here since network policies are typically few in number (unlike users).
from snowflake.snowpark.context import get_active_session
import pandas as pd

session = get_active_session()

policies = session.sql("SHOW NETWORK POLICIES").collect()

rows = []
for p in policies:
    policy_name = p["name"]
    try:
        detail = session.sql(f'DESCRIBE NETWORK POLICY "{policy_name}"').collect()
        for d in detail:
            rows.append({
                "policy_name": policy_name,
                "property": d["name"] if "name" in d.as_dict() else None,
                "value": d["value"] if "value" in d.as_dict() else None,
            })
    except Exception as e:
        rows.append({"policy_name": policy_name, "property": "ERROR", "value": str(e)})

result_df = pd.DataFrame(rows)
result_df


## 2. Storage & other integrations

Every integration is a trust boundary — it lets Snowflake talk to something outside itself.
Each one is worth a deliberate "why does this exist and who owns it" answer, not an assumption
that it's all fine because it's already there.


In [ ]:
-- All integrations, of every category (storage, API, notification, security, external access)
SHOW INTEGRATIONS;


In [ ]:
-- Storage integrations specifically, with their allowed/blocked locations
SHOW STORAGE INTEGRATIONS;


For each storage integration that looks broad (an `STORAGE_ALLOWED_LOCATIONS` value of a
whole bucket root rather than a specific prefix, or an empty `STORAGE_BLOCKED_LOCATIONS`),
run `DESCRIBE STORAGE INTEGRATION <name>;` individually to see the full location list and
confirm it's scoped as narrowly as the pipeline actually needs.


In [ ]:
-- External functions and API integrations they depend on — another data egress path,
-- since an external function call sends data to code running outside Snowflake.
SHOW EXTERNAL FUNCTIONS;


## 3. External stages

Internal (Snowflake-managed) stages keep data inside Snowflake's control boundary. External
stages point at customer-managed cloud storage — that's not inherently wrong, but it's where
"data leaving Snowflake" actually happens, so it's worth knowing exactly how many there are
and who can write to them.


In [ ]:
-- All external stages in the account
SELECT
    stage_catalog,
    stage_schema,
    stage_name,
    stage_url,
    stage_type,
    created,
    last_altered
FROM snowflake.account_usage.stages
WHERE deleted IS NULL
  AND stage_type = 'External Named'
ORDER BY stage_catalog, stage_schema, stage_name;


In [ ]:
-- Who can write to each external stage (WRITE or OWNERSHIP privilege)
SELECT
    g.grantee_name AS role_name,
    g.privilege,
    g.name         AS stage_name,
    g.table_catalog AS database_name,
    g.table_schema AS schema_name
FROM snowflake.account_usage.grants_to_roles g
WHERE g.deleted_on IS NULL
  AND g.granted_on = 'STAGE'
  AND g.privilege IN ('WRITE', 'OWNERSHIP', 'ALL PRIVILEGES')
ORDER BY stage_name, role_name;


## 4. Data unloading / egress controls

These three account parameters are the primary account-wide levers for whether data can leave
Snowflake via `COPY INTO <location>` at all. In a confidentiality-sensitive environment, the
default answer to "should unrestricted unload be allowed" is usually no — egress should
require a defined storage integration with a known, scoped destination, not an arbitrary
inline URL or credentials pasted into a query.


In [ ]:
-- Account-wide unload/egress control parameters
SHOW PARAMETERS LIKE 'PREVENT_UNLOAD%' IN ACCOUNT;


In [ ]:
-- Whether stages must use a storage integration (vs. inline cloud credentials in the CREATE STAGE statement)
SHOW PARAMETERS LIKE 'REQUIRE_STORAGE_INTEGRATION%' IN ACCOUNT;


In [ ]:
-- Recent actual unload activity: COPY INTO statements targeting an external/internal stage
-- rather than a table (heuristic on query_text, since query_type alone doesn't cleanly
-- distinguish "load" from "unload" across all Snowflake releases).
SELECT
    query_id,
    user_name,
    role_name,
    warehouse_name,
    start_time,
    LEFT(query_text, 250) AS query_text_preview,
    rows_produced,
    bytes_scanned
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (query_text ILIKE 'COPY INTO @%' OR query_text ILIKE 'COPY INTO ''%')
ORDER BY start_time DESC
LIMIT 200;


In [ ]:
-- Same window, rolled up by user/warehouse so a pattern (one account doing this routinely
-- vs. a one-off) is easy to spot
SELECT
    user_name,
    warehouse_name,
    COUNT(*) AS unload_statement_count,
    MIN(start_time) AS first_seen,
    MAX(start_time) AS last_seen
FROM snowflake.account_usage.query_history
WHERE start_time >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND execution_status = 'SUCCESS'
  AND (query_text ILIKE 'COPY INTO @%' OR query_text ILIKE 'COPY INTO ''%')
GROUP BY user_name, warehouse_name
ORDER BY unload_statement_count DESC;


## 5. Data sharing inventory

Outbound shares are a direct, sanctioned data-egress path — worth a full inventory of what's
shared, to whom, and whether it's still needed, rather than assuming every share still in
existence is still intentional.


In [ ]:
-- All shares, inbound and outbound
SHOW SHARES;


In [ ]:
-- What's actually exposed through each outbound share
SELECT
    grantee_name AS share_name,
    privilege,
    granted_on   AS object_type,
    name         AS object_name,
    table_catalog AS database_name,
    created_on
FROM snowflake.account_usage.grants_to_roles
WHERE deleted_on IS NULL
  AND grantee_name IN (
      -- shares appear in grants_to_roles with the share name as grantee once objects are granted to it
      SELECT DISTINCT grantee_name FROM snowflake.account_usage.grants_to_roles
  )
ORDER BY share_name, object_type;


If the query above returns more than expected, narrow it to specific share names from the
`SHOW SHARES` output — `grants_to_roles` doesn't have a clean `is_share` flag across all
Snowflake releases, so this may need a `WHERE grantee_name IN ('SHARE_NAME_1', 'SHARE_NAME_2')`
filter using the actual share names from this environment.


## 6. Masking & row access policy coverage

For a healthcare/confidential-data environment, this section matters more than any other in
the notebook: are the columns that actually hold sensitive data protected by a policy, or
does protection depend entirely on RBAC (i.e., anyone with `SELECT` sees everything
unmasked)?


In [ ]:
-- Masking policies defined in the account
SELECT
    policy_db AS masking_policy_catalog,
    policy_schema AS masking_policy_schema,
    policy_name AS masking_policy_name,
    policy_kind,
    ref_database_name,
    ref_schema_name,
    ref_entity_name,
    ref_column_name,
    policy_status
FROM snowflake.account_usage.policy_references
WHERE policy_kind = 'MASKING_POLICY'
ORDER BY ref_database_name, ref_schema_name, ref_entity_name;


In [ ]:
-- Row access policies defined and where applied
SELECT
    policy_db AS policy_catalog,
    policy_schema,
    policy_name,
    policy_kind,
    ref_database_name,
    ref_schema_name,
    ref_entity_name,
    policy_status
FROM snowflake.account_usage.policy_references
WHERE policy_kind = 'ROW_ACCESS_POLICY'
ORDER BY ref_database_name, ref_schema_name, ref_entity_name;


In [ ]:
-- Columns that LOOK sensitive by name but have no masking policy applied anywhere.
-- This is the single highest-value query in this notebook for a healthcare environment.
WITH masked_columns AS (
    SELECT DISTINCT
        ref_database_name AS database_name,
        ref_schema_name   AS schema_name,
        ref_entity_name   AS table_name,
        ref_column_name   AS column_name
    FROM snowflake.account_usage.policy_references
    WHERE policy_kind = 'MASKING_POLICY'
      AND policy_status = 'ACTIVE'
)
SELECT
    c.table_catalog AS database_name,
    c.table_schema  AS schema_name,
    c.table_name,
    c.column_name,
    c.data_type
FROM snowflake.account_usage.columns c
LEFT JOIN masked_columns m
    ON c.table_catalog = m.database_name
   AND c.table_schema  = m.schema_name
   AND c.table_name    = m.table_name
   AND c.column_name   = m.column_name
WHERE c.deleted IS NULL
  AND c.table_schema != 'INFORMATION_SCHEMA'
  AND m.column_name IS NULL
  AND EXISTS (
        SELECT 1 FROM TABLE(SPLIT_TO_TABLE($sensitive_column_patterns, ',')) p
        WHERE c.column_name ILIKE p.value
      )
ORDER BY database_name, schema_name, table_name, column_name;


In [ ]:
-- Secure vs. non-secure views — a non-secure view over sensitive base tables can leak
-- the underlying query logic (and, via query optimization, sometimes data) to viewers who
-- shouldn't see the base table directly.
SELECT
    table_catalog AS database_name,
    table_schema,
    table_name AS view_name,
    is_secure,
    view_owner
FROM snowflake.account_usage.views
WHERE deleted IS NULL
  AND table_schema != 'INFORMATION_SCHEMA'
  AND is_secure = 'NO'
ORDER BY database_name, table_schema, view_name;


## 7. Authentication & session security

Configuration alone isn't proof of practice — Section 4's login-history query shows what
authentication factors are actually being used, not just what's theoretically available.


In [ ]:
-- Password and session policies defined, and where (if anywhere) they're attached
SHOW PASSWORD POLICIES;


In [ ]:
-- Authentication policies (MFA enforcement, allowed auth methods) — newer Snowflake accounts
SHOW AUTHENTICATION POLICIES;


In [ ]:
-- MFA usage in practice across all human users in the recent window
SELECT
    second_authentication_factor,
    COUNT(*) AS login_count,
    COUNT(DISTINCT user_name) AS distinct_users
FROM snowflake.account_usage.login_history
WHERE event_timestamp >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND is_success = TRUE
GROUP BY second_authentication_factor
ORDER BY login_count DESC;


In [ ]:
-- Individual users who never use a second factor, for direct follow-up
SELECT
    user_name,
    COUNT(*) AS login_count,
    MAX(event_timestamp) AS last_login
FROM snowflake.account_usage.login_history
WHERE event_timestamp >= DATEADD(day, -$lookback_days, CURRENT_TIMESTAMP())
  AND is_success = TRUE
GROUP BY user_name
HAVING SUM(IFF(second_authentication_factor IS NOT NULL, 1, 0)) = 0
ORDER BY login_count DESC;


## 8. Rolled-up recommendation summary

This one is a checklist rather than a query — the sections above return object-level detail,
but "is the account secure" is ultimately a set of yes/no configuration questions plus the
sensitive-column-coverage finding from Section 6. Fill this in from the results above.

| Area | Finding | Risk if unaddressed | Status |
|---|---|---|---|
| Account-level network policy enforced | *from Section 1* | Any authenticated user can connect from anywhere | |
| Storage integrations scoped narrowly | *from Section 2* | Overly broad cloud storage access if credentials are ever misused | |
| External stages inventoried & write access reviewed | *from Section 3* | Unclear/uncontrolled data egress paths | |
| `PREVENT_UNLOAD_TO_INLINE_URL` / `PREVENT_UNLOAD_TO_INTERNAL_STAGES` set appropriately | *from Section 4* | Unrestricted ad hoc data unload | |
| Outbound shares reviewed and still-needed | *from Section 5* | Sanctioned but forgotten egress path | |
| Sensitive-looking columns have masking policies applied | *from Section 6* | Confidential/PHI-type data visible to anyone with SELECT | |
| Views over sensitive data are SECURE | *from Section 6* | View logic/data exposure to unintended viewers | |
| MFA actually used, not just available | *from Section 7* | Password-only access to sensitive data | |


### Notes, caveats, and next steps

- **Run this alongside the RBAC audit notebook.** Section 6's unmasked-sensitive-column list
  is most actionable cross-referenced with *who* can currently `SELECT` those columns — a
  column with no masking policy but also very few roles granted access is a lower priority
  than one with no masking policy and broad `SELECT` access.
- **The sensitive-column name matching is a heuristic, not a discovery tool.** It will miss
  sensitive columns with non-obvious names and may over-flag ordinary columns (e.g., an
  `EMAIL` column on an internal contacts table that was never meant to be PHI-sensitive).
  Confirm findings with the data owner before treating this as a definitive gap list — and if
  Snowflake's classification/tagging features are enabled in this account, that's a stronger
  source of truth than name-matching and worth switching to once available.
- **Tri-Secret Secure / customer-managed encryption keys** aren't queryable from
  `ACCOUNT_USAGE` — that's an edition/contract-level setting to confirm directly with the
  account team if end-to-end encryption key ownership matters for this environment's
  compliance requirements.
- **This is a snapshot.** For a regulated environment, the real goal is usually to get from
  "we ran this once" to "this runs on a schedule and alerts on drift" — once the first pass is
  clean, consider scheduling the unload-activity and MFA-usage queries as a recurring task
  with alerting on new findings, rather than only running this manually.
